# Local Qwen/vLLM workflow

This template serves a vision-language model through vLLM's OpenAI-compatible API, then runs PaperMiner against it. Adapt the model, device, dtype, tensor parallelism, and context length to your hardware.

In [ ]:
%env PS_DB=papers.db
%env PS_QUERY=lithium solid electrolyte
%env PS_RECIPE=sse
%env PS_LOCAL_MODEL=Qwen/Qwen3-VL-30B-A3B-Instruct
%env PS_BASE_URL=http://127.0.0.1:8000/v1
%env PS_MAX_MODEL_LEN=120000
%env PS_OUTPUT=temp_qwen_materials.csv
%env PS_FINAL=qwen_materials.csv

## Start and verify the server

Run this only in an accelerator allocation with a compatible vLLM installation. The context length must fit the available memory.

In [ ]:
import os
import subprocess
import time

import requests

log = open("vllm.log", "w", encoding="utf-8")
server = subprocess.Popen(
    [
        "vllm", "serve", os.environ["PS_LOCAL_MODEL"],
        "--host", "127.0.0.1", "--port", "8000",
        "--max-model-len", os.environ["PS_MAX_MODEL_LEN"],
    ],
    stdout=log, stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        response = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        response.raise_for_status()
        break
    except requests.RequestException:
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not become ready; inspect vllm.log")

## Configure PaperMiner

The local provider does not require a hosted-model API key. Both profiles point at the same server because this Qwen model accepts text and image input.

In [ ]:
%%bash
set -euo pipefail
ps_model_config text --provider local --model "$PS_LOCAL_MODEL" --base-url "$PS_BASE_URL" --input-token-limit "$PS_MAX_MODEL_LEN"
ps_model_config vision --provider local --model "$PS_LOCAL_MODEL" --base-url "$PS_BASE_URL" --input-token-limit "$PS_MAX_MODEL_LEN"
ps_model_status

## Build, scrape, and store

Corpus discovery and downloading do not use the local model. Keep those stages off scarce accelerator resources when possible.

In [ ]:
%%bash
set -euo pipefail
ps_search "$PS_QUERY" "$PS_DB" --source openalex --count 25
ps_download "$PS_DB" --format both
ps_scrape "$PS_DB" "$PS_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PS_OUTPUT"
ps_store "$PS_DB" "$PS_OUTPUT" "$PS_FINAL" "$PS_RECIPE" --assume-yes